In [11]:
# test forecast
import sys
import os
import numpy as np
from scipy import stats
import json

from itertools import combinations

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import OccurencesStats
from stats_lib import DelaysStats
from stats_lib import SequenceStats

######################
def get_occurrences_stats(evidenze):
    occurrences = evidenze.get('occurrences', {})
    
    limits = [] * len(occurrences)
    for key, class_data in occurrences.items(): 
        k = class_data.get("k", int(key))
        l_class = {}
        l_class["sample"] = k
        l_class["class"] = [{}] * len(class_data)
        for c_key, c_value in class_data.items():
            #print(f"  {c_key}")
            desc = c_value.get("description", "")
            measures = c_value.get("measures", {})
            values = measures.get("values", [])
            
            #print(f"\nClasse k={k}: {c_value} {desc}")
            s_limit = {}
            s_limit["k"] = c_key
            s_limit["bounds"] = [{}] * 4 
            for measure in values:
                m_id = measure.get("id", "")
                m_type = measure.get("type", "")
                m_mean = measure.get("mean", [])
                m_std = measure.get("std", [])
                m_skew = measure.get("skewness", [])
                m_q1 = measure.get("q1", [])
                m_q3 = measure.get("q3", [])
                
                # print(f"  {m_type}:")
                # print(f"    mean    : {m_mean}")
                # print(f"    std     : {m_std}")
                # print(f"    skewness: {m_skew}")
                # print(f"    q1      : {m_q1}")
                # print(f"    q3      : {m_q3}")

                simm = measure.get("simmetrica", {})
                asimm = measure.get("asimmetrica", {})
                
                bounds = {}
                t = abs(m_skew)
                match t:
                    case t if t < 0.5:
                        #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                        bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                    case t if t > 1.0:
                        #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                        bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
                    case _:
                        if simm.get('coverage') >= asimm.get('coverage'):
                            bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                            #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                        else:
                            #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                            bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
                
                #print(f"    bounds   : min={bounds.get('min')}, max={bounds.get('max')}")
                s_limit["bounds"][int(m_id)] = bounds
                
            #print(f"s_limit: {s_limit}")
        
            l_class['class'][int(c_key)-1] = s_limit
            #print(f"l_class: {l_class}")
            
        limits = limits + [l_class]
        

    return limits

def get_base_occurrences(base_combinations, sample):
    occurrences = [[] * 4] * 6
    for k in range(1,7):
        o = ostats.single_combination_occurrences(base_combinations[k-1], sample)
        #print(f"Occurrences for {base}: {o} , k = {k}")
        tot = np.sum(o)
        nz = np.count_nonzero(o)
        z = len(base_combinations[k-1]) - nz
        m = np.max(o)
        #print(f"Total occurrences: {tot} - Non-zero occurrences: {nz} - Zero occurrences: {z} - Max occurrences: {m}")
        occurrences[k-1] = [tot, nz, z, m]
        
    return occurrences

def check_occurrences(base_combinations, history_data, limits):

    for o in limits:
        sample_size = int(o.get('sample'))
        sample = history_data[:sample_size] 
        occurrences = get_base_occurrences(base_combinations, sample)
        # print(f"Occurrences: {occurrences}")
        # print(f"Sample: {sample_size}")
        classes = o.get('class', [])
        for c in classes:
            #print(f"- Class: {c.get('k')}")
            bounds = c.get('bounds', [])
            occ = occurrences[int(c.get('k'))-1]
            #print(f"  - Occurrences for class {c.get('k')}: {occ}")
            for b in bounds:
                #print(f"  - bound: {b}")
                id = int(b.get('id'))
                t = occ[id]
                #print(f"  - Bounds for measure={id}: min={b.get('min')}, max={b.get('max')}, current forecast={t}")
                if t < b.get('min') or t > b.get('max'):
                    ##print(f"    - Measure {id} is outside the bounds!")
                    return True, {"Sample": sample_size, "Class": c.get('k'), "Measure": id, "Bounds": {"min": b.get('min'), "max": b.get('max'), "current": t}} # Log the first measure that is outside the bounds
                    
    return False, {} # All measures are within the bounds (i.e. no waste)

# def get_attempts(total_attempts, clib):
#     attempts = [[]] * total_attempts
#     for i in range(total_attempts):
#         attempts[i] = clib.get_forecast()
#     return attempts

def get_attempts(total_attempts, get_forecast):
    attempts = [[]] * total_attempts
    for i in range(total_attempts):
        attempts[i] = get_forecast()
    return attempts

def collect_attempts_within_occurrences(attempts, history_data, limits):
    good_attempts = [[]] * len(attempts)
    j = 0 # Index for good attempts
    for i in range(len(attempts)):
        print(f"Checking attempt {i+1}/{len(attempts)}", end="\r")
        base = attempts[i]
        base_combinations = [list(combinations(base, k)) for k in range(1,7)]
        waste, log_waste = check_occurrences(base_combinations, history_data, limits)
        if not waste:
            good_attempts[j] = base
            j += 1
            
    return [row for row in good_attempts if len(row) > 0]

def get_sequences_stats(evidenze):
    values = evidenze.get('sequences', {}).get('measures', {}).get('values')
    #print("tot values: ", len(values))
    
    s_limit = {}
    s_limit["bounds"] = [{}] * len(values) 
    for value in values: 
        m_id = value.get("id", "")
        m_type = value.get("type", "")
        m_mean = value.get("mean", [])
        m_std = value.get("std", [])
        m_skew = value.get("skewness", [])
        m_q1 = value.get("q1", [])
        m_q3 = value.get("q3", [])

        simm = value.get("simmetrica", {})
        asimm = value.get("asimmetrica", {})
                
        bounds = {}
        t = abs(m_skew)
        match t:
            case t if t < 0.5:
                #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
            case t if t > 1.0:
                #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
            case _:
                if simm.get('coverage') >= asimm.get('coverage'):
                    bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                    #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                else:
                    #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                    bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
        
        #print(f"    bounds {m_id}  : min={bounds.get('min')}, max={bounds.get('max')}")
        s_limit["bounds"][int(m_id)] = bounds
        #print("s_limit: ", m_id, " ", s_limit)
            
    #limits = limits + [s_limit]

    return s_limit
           
def check_sequences(base, limits):
    seq = sstats.count_sequences(base)

    bounds = limits.get('bounds', [])
    for b in bounds:
        #print(f"  - bound: {b}")
        id = int(b.get('id'))
        t = seq[id]
        #print(f"  - Bounds for measure={id}: min={b.get('min')}, max={b.get('max')}, current forecast={t}")
        if t < b.get('min') or t > b.get('max'):
            ##print(f"    - Measure {id} is outside the bounds!")
            return True, {"Measure": id, "Bounds": {"min": b.get('min'), "max": b.get('max'), "current": t}} # Log the first measure that is outside the bounds
                
    return False, {} # All measures are within the bounds (i.e. no waste)

def collect_attempts_within_sequences(attempts, limits):
    good_attempts = [[]] * len(attempts)
    j = 0 # Index for good attempts
    for i in range(len(attempts)):
        print(f"Checking attempt {i+1}/{len(attempts)}", end="\r")
        base = attempts[i]
        waste, log_waste = check_sequences(base, limits)
        if not waste:
            good_attempts[j] = base
            j += 1
            
    return [row for row in good_attempts if len(row) > 0]


def get_delays_stats(evidenze):
    values = evidenze.get('delays', {}).get('measures', {}).get('values')
    #print("tot values: ", len(values))
    
    d_limit = {}
    d_limit["bounds"] = [{}] * len(values) 
    for value in values: 
        m_id = value.get("id", "")
        m_type = value.get("type", "")
        m_mean = value.get("mean", [])
        m_std = value.get("std", [])
        m_skew = value.get("skewness", [])
        m_q1 = value.get("q1", [])
        m_q3 = value.get("q3", [])

        simm = value.get("simmetrica", {})
        asimm = value.get("asimmetrica", {})
                
        bounds = {}
        t = abs(m_skew)
        match t:
            case t if t < 0.5:
                #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
            case t if t > 1.0:
                #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
            case _:
                if simm.get('coverage') >= asimm.get('coverage'):
                    bounds = {"id": m_id, "min": simm.get('min'), "max": simm.get('max')}
                    #print(f"simmetrica   {t}: min={simm.get('min')}, max={simm.get('max')}, coverage={simm.get('coverage')}%")
                else:
                    #print(f"asimmetrica  {t}: min={asimm.get('min')}, max={asimm.get('max')}, coverage={asimm.get('coverage')}%")
                    bounds = {"id": m_id, "min": asimm.get('min'), "max": asimm.get('max')}
        
        #print(f"    bounds {m_id}  : min={bounds.get('min')}, max={bounds.get('max')}")
        d_limit["bounds"][int(m_id)] = bounds
        #print("d_limit: ", m_id, " ", d_limit)
            
    #limits = limits + [s_limit]

    return d_limit

def check_delays(base, sample, limits):
    dels = dstats.count_delays(base, sample)
    #print("Base: ", base, " - ritardi: ", dels)
    bounds = limits.get('bounds', [])
    for b in bounds:
        #print(f"  - bound: {b}")
        id = int(b.get('id'))
        t = dels[id]
        #print(f"  - Bounds for measure={id}: min={b.get('min')}, max={b.get('max')}, current forecast={t}")
        if t < b.get('min') or t > b.get('max'):
            #print(f"    - Measure {id} is outside the bounds!")
            print(f"Outside bounds for measure={id}: min={b.get('min')}, max={b.get('max')}, current forecast={t}")
            return True, {"Measure": id, "Bounds": {"min": b.get('min'), "max": b.get('max'), "current": t}} # Log the first measure that is outside the bounds
                
    return False, {} # All measures are within the bounds (i.e. no waste)

def collect_attempts_within_delays(attempts, sample, limits):
    good_attempts = [[]] * len(attempts)
    j = 0 # Index for good attempts
    for i in range(len(attempts)):
        print(f"Checking attempt {i+1}/{len(attempts)}", end="\r")
        base = attempts[i]
        waste, log_waste = check_delays(base, sample, limits)
        if not waste:
            good_attempts[j] = base
            j += 1
            
    return [row for row in good_attempts if len(row) > 0]

######################
# Caricamento regole di validazine
evidenze = {}
with open('../collector_results.json') as file_evidenze:
        evidenze = json.load(file_evidenze)
        
# Popolazione        
data_file_name = "../data/estrazioni_2026_2006_copy.csv"
clib = CommonLib(data_file_name)
history_data = clib.load_data() # Carica le estrazioni da un file CSV

ostats = OccurencesStats()
sstats = SequenceStats()
dstats = DelaysStats()

o_limits = get_occurrences_stats(evidenze)
s_limits = get_sequences_stats(evidenze)
d_limits = get_delays_stats(evidenze)
#print("delay: ", d_limits)

#
# 20260131;2;6;7;33;37;80
# 20260130;32;33;39;40;52;86
# 20260129;29;30;34;56;66;80
# 20260127;11;19;27;31;54;84
# 20260124;22;37;55;61;68;71
# 20260123;2;6;11;19;88;90
# 20260122;2;30;52;56;57;78
p = [0] * 7
p [0] = [2,30,52,56,57,78]
p [1] = [2,6,11,19,88,90]
p [2] = [22,37,55,61,68,71]
p [3] = [11,19,27,31,54,84]
p [4] = [29,30,34,56,66,80]
p [5] = [32,33,39,40,52,86]
p [6] = [2,6,7,33,37,80]

extracted_base = p[0]

total_attempts = 100
attempts = get_attempts(total_attempts, clib.get_forecast)

print("Checking attempts within occurrences")
good_attempts = collect_attempts_within_occurrences(attempts, history_data, o_limits)


if len(good_attempts) > 0:
    print("\nAttempts passed: ", len(good_attempts))
    
    print("\nChecking attempts within sequences")
    attempts = good_attempts
    good_attempts = collect_attempts_within_sequences(attempts, s_limits)
    
    if len(good_attempts) > 0:
        print("\nAttempts passed: ", len(good_attempts))
            
        print("\nChecking attempts within delays")
        attempts = good_attempts
        hyp_max_delay = clib.get_hypothetic_max_delay()
        sample = history_data[0:hyp_max_delay]
        good_attempts = collect_attempts_within_delays(attempts, sample, d_limits)
        
        if len(good_attempts) > 0:
            print("\nAttempts passed: ", len(good_attempts))
        else:
            print("\nNo good attempts found")
    else:
        print("\nNo good attempts found")
        
        
else:
    print("\nNo good attempts found")



################# find good attempts matching actual extracted base ###########################
#    
s = total_attempts - len(good_attempts) # Counter for attempts with at least one measure outside bounds
if total_attempts - s > 0:
    print(f"\nTotal attempts: {total_attempts} - Attempts with at least one measure outside bounds: {s} - Good attempts: {total_attempts - s}")
    #print ("Good attempts:", good_attempts)
    print("Extracted base:", extracted_base)
    count_attempts_with_match = 0
    for attempt in good_attempts:  
        count_matches = np.count_nonzero(attempt == extracted_base)
        if count_matches > 0:
            count_attempts_with_match += 1
            print("Matching attempt:", attempt, " - " , count_matches, " attempt item(s) matched extracted base:") 
    if count_attempts_with_match == 0:
        print("No matching attempts")        
else:
    print("\nNo good attempts found")


Checking attempts within occurrences
Checking attempt 100/100
Attempts passed:  8

Checking attempts within sequences
Checking attempt 8/8
Attempts passed:  8

Checking attempts within delays
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=5
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=16
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=52
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=5
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=35
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=13
Outside bounds for measure=1: min=22.0, max=44.0, current forecast=10
Outside bounds for measure=0: min=0.0, max=3.0, current forecast=9

No good attempts found

No good attempts found


In [ ]:

# test produzione pronostici dai ritardi
import sys
import os
import numpy as np
from scipy import stats
import json
import random as rnd

from itertools import combinations

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import OccurencesStats
from stats_lib import DelaysStats
from stats_lib import SequenceStats

######################
# delays:  {'bounds': [{'id': 0, 'min': 0.0, 'max': 3.0}, {'id': 1, 'min': 22.0, 'max': 44.0}, {'id': 2, 'min': 8.0, 'max': 20.0}, {'id': 3, 'min': 50.0, 'max': 118.0}]}
# dentro bound ci devono essere i max di ogni colonna dello storico
def find_attempts(idx_a, bounds, attempts, history_data, delays):
    if idx_a < 6:
        valid = False
        # search for valid attempt
        #print("ricerca pronostico: ", idx_a)
        while not valid:
            amin, amax = get_limits(idx_a, bounds, attempts, delays)
            #print("limiti: ", amin, amax)
            attempt = rnd.randint(amin, amax)
            #print("prova: ", attempt)
            d = count_single_delay(attempt, history_data) 
            # d[0] must be within min of minimum delays and max of maximum delays #
            d_min = delays.get("bounds")[0].get("min")
            d_max = delays.get("bounds")[1].get("max")
            if d_min <= d <= d_max:
               #print("valido ", attempts, d, d_min, d_max)
               attempts.append(attempt)
               # Next attempt
               idx_a += 1
               find_attempts(idx_a, bounds, attempts, history_data, delays)
               # valid attempt, break retry
               break
            # retry attempt 
            # print("non valido")
    else:
        # Pronostici completati
        exit
        
def count_single_delay(attempt, history_data):
    hd_size = len(history_data)
    delay = 0
    for i in range(hd_size):
        if attempt not in history_data[i]: 
            delay += 1
        else: break
    
    return delay
            
def get_limits(idx_a, bounds, attempts, delays):
    amin = attempts[idx_a - 1] + 1 if idx_a > 0 else 1
    amax = bounds[idx_a]
    return amin, amax
       
def validate_attempts(attempts, delays, history_data):
    d = dstats.count_delays(attempts, history_data)
    dd = [min(d), max(d), np.round(np.mean(d), 0), np.sum(d)]
    #print("d  ", d)
    #print("dd ", dd)
    tr = ["ritardo minimo", "ritardo massimo", "ritardo medio", "ritardo totale"]
    for k in range(4):
        if dd[k] < delays.get("bounds")[k].get("min") or delays.get("bounds")[k].get("max") < dd[k]:
            #print(f"pronostico non valido - {tr[k]} {dd[k]} fuori limiti {delays.get("bounds")[k].get("min")}:{delays.get("bounds")[k].get("max")}")
            return False
    return True
        

######################
# Caricamento regole di validazine
evidenze = {}
with open('../collector_results.json') as file_evidenze:
        evidenze = json.load(file_evidenze)
        
# Popolazione        
data_file_name = "../data/estrazioni_2026_2006_copy.csv"
clib = CommonLib(data_file_name)
history_data = clib.load_data() # Carica le estrazioni da un file CSV

dstats = DelaysStats()

delays = get_delays_stats(evidenze)


bounds = np.asarray(history_data).max(axis=0, keepdims=True)[0]
#print("Bounds: ", bounds)

i = 0
ja = 0
js = 0
at = [[] * 6] * 10
while i < 10:
    attempts = [] * 6
    find_attempts(0, bounds, attempts, history_data, delays)
    #print("Pronostico: ", attempts)
    valido = validate_attempts(attempts, delays, history_data)
    if valido: 
        #print(attempts, " pronostico accettato")
        at[i] = attempts
        i += 1
        ja += 1
    else:
        #print(attempts, "pronostico scartato")
        js += 1
        
print(f"{ja + js} pronostici generati - accettati {ja} - scartati {js}")
for i in range(10):
    print("Pronostico: ", at[i])

#[2,30,52,56,57,78]


27 pronostici generati - accettati 10 - scartati 17
Pronostico:  [1, 19, 54, 82, 85, 90]
Pronostico:  [54, 72, 76, 78, 85, 90]
Pronostico:  [3, 62, 66, 86, 89, 90]
Pronostico:  [10, 76, 81, 86, 88, 89]
Pronostico:  [38, 47, 70, 73, 82, 84]
Pronostico:  [59, 77, 78, 80, 84, 89]
Pronostico:  [54, 60, 65, 84, 89, 90]
Pronostico:  [30, 53, 62, 74, 78, 89]
Pronostico:  [54, 82, 84, 85, 89, 90]
Pronostico:  [26, 69, 82, 84, 89, 90]


In [13]:
################# find good attempts matching actual extracted base ###########################
#    
# test forecast
import sys
import os
import numpy as np
from scipy import stats
import json

from itertools import combinations

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import OccurencesStats
from stats_lib import DelaysStats
from stats_lib import SequenceStats

p = [0] * 7
p [0] = [2,30,52,56,57,78]
p [1] = [2,6,11,19,88,90]
p [2] = [22,37,55,61,68,71]
p [3] = [11,19,27,31,54,84]
p [4] = [29,30,34,56,66,80]
p [5] = [32,33,39,40,52,86]
p [6] = [2,6,7,33,37,80]

extracted_base = p[0]

forecast_file = "../forecast.csv"
forecast = np.loadtxt(
            forecast_file, 
            dtype=int, 
            delimiter=';'
        ) 

#total_attempts = len(forecast) 
print("Extracted base:", extracted_base)
count_attempts_with_match = 0
for attempt in forecast:  
    count_matches = np.count_nonzero(attempt == extracted_base)
    if count_matches > 0:
        count_attempts_with_match += 1
        print("Matching attempt:", attempt, " - " , count_matches, " attempt item(s) matched extracted base:") 
if count_attempts_with_match == 0:
    print("No matching attempts")        


Extracted base: [2, 30, 52, 56, 57, 78]
Matching attempt: [ 8 23 52 64 69 90]  -  1  attempt item(s) matched extracted base:
Matching attempt: [19 36 52 62 66 69]  -  1  attempt item(s) matched extracted base:
Matching attempt: [ 1 30 37 51 79 83]  -  1  attempt item(s) matched extracted base:
Matching attempt: [13 33 52 61 70 77]  -  1  attempt item(s) matched extracted base:
Matching attempt: [14 16 41 62 70 78]  -  1  attempt item(s) matched extracted base:
Matching attempt: [ 2 22 46 65 69 84]  -  1  attempt item(s) matched extracted base:
Matching attempt: [ 8 36 44 46 67 78]  -  1  attempt item(s) matched extracted base:
Matching attempt: [ 2 37 49 58 72 84]  -  1  attempt item(s) matched extracted base:
Matching attempt: [ 7 36 52 59 67 88]  -  1  attempt item(s) matched extracted base:
Matching attempt: [12 35 52 62 66 81]  -  1  attempt item(s) matched extracted base:
Matching attempt: [15 24 52 62 72 73]  -  1  attempt item(s) matched extracted base:
Matching attempt: [17 37 

In [6]:
################# test row sum ###########################
#    
# test forecast
import sys
import os
import numpy as np
from scipy import stats
import json

from itertools import combinations

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import OccurencesStats
from stats_lib import DelaysStats
from stats_lib import SequenceStats

evidenze = {}
with open('../collector_results.json') as file_evidenze:
        evidenze = json.load(file_evidenze)

rowsums = evidenze.get('row_sums', {}).get('measures', {}).get('values', [])
print("Row sums: ", rowsums.get("min"), " - ", rowsums.get("max"), " - ", rowsums.get("mean"), " - ", rowsums.get("std_dev"))

Row sums:  81  -  467  -  276.23  -  61.23
